<a href="https://colab.research.google.com/github/kapilvarunbabu-g/ai-projects/blob/main/interviewchatbot/interviewchatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║      AI INTERVIEW ASSISTANT — Google Colab Version                   ║
# ║      Copy each cell block into a separate Colab cell                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# HOW TO USE IN GOOGLE COLAB:
# 1. Go to https://colab.research.google.com
# 2. Create a new notebook
# 3. Copy each "# ── CELL X" block into its own Colab cell
# 4. Run cells top to bottom (Shift+Enter)
# 5. Click the public gradio.live link that appears


# ══════════════════════════════════════════════════════════════════════
# ── CELL 1 ── INSTALL DEPENDENCIES
# ══════════════════════════════════════════════════════════════════════

# !pip install -q gradio spacy
# !python -m spacy download en_core_web_sm -q


# ══════════════════════════════════════════════════════════════════════
# ── CELL 2 ── IMPORTS & NLP SETUP
# ══════════════════════════════════════════════════════════════════════

import gradio as gr
import json, os, random, re, time, socket
from datetime import datetime

os.makedirs("data", exist_ok=True)
RESULTS_FILE = "data/results.json"

try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    SPACY_AVAILABLE = True
    print("✅ spaCy loaded successfully")
except Exception:
    SPACY_AVAILABLE = False
    nlp = None
    print("⚠️  spaCy unavailable — using keyword fallback")


# ══════════════════════════════════════════════════════════════════════
# ── CELL 3 ── QUESTION DATABASE (45 Questions)
# ══════════════════════════════════════════════════════════════════════

QUESTION_BANK = [
    # ── TECHNICAL (15) ───────────────────────────────────────────────
    {"id":"T01","category":"Technical","difficulty":"easy",
     "question":"What is a Python list comprehension? Give an example.",
     "correct_answer":"A list comprehension is a concise way to create lists using a single line syntax. Example: [x**2 for x in range(10)] creates a list of squares.",
     "expected_keywords":["list","comprehension","syntax","concise","loop","expression","square bracket"],"marks":5},

    {"id":"T02","category":"Technical","difficulty":"medium",
     "question":"Explain the difference between a shallow copy and a deep copy in Python.",
     "correct_answer":"A shallow copy creates a new object but references the same nested objects. A deep copy creates a completely independent clone including all nested objects.",
     "expected_keywords":["shallow","deep","copy","nested","reference","independent","clone","object"],"marks":5},

    {"id":"T03","category":"Technical","difficulty":"medium",
     "question":"What are Python decorators and how do they work?",
     "correct_answer":"Decorators are functions that modify the behavior of another function without changing its source code. They use the @syntax and wrap the target function.",
     "expected_keywords":["decorator","function","wrap","modify","behavior","syntax","@","higher-order"],"marks":5},

    {"id":"T04","category":"Technical","difficulty":"hard",
     "question":"Explain Python's GIL (Global Interpreter Lock) and its impact on multithreading.",
     "correct_answer":"The GIL is a mutex that protects Python objects, preventing multiple threads from executing Python bytecode simultaneously. It limits true parallelism in CPU-bound threads.",
     "expected_keywords":["GIL","mutex","thread","parallel","bytecode","CPU","I/O","lock","interpreter"],"marks":10},

    {"id":"T05","category":"Technical","difficulty":"medium",
     "question":"What is the difference between SQL INNER JOIN, LEFT JOIN, and RIGHT JOIN?",
     "correct_answer":"INNER JOIN returns rows matching in both tables. LEFT JOIN returns all left rows + matching right rows (NULL if no match). RIGHT JOIN is the reverse.",
     "expected_keywords":["inner join","left join","right join","matching","null","rows","table"],"marks":5},

    {"id":"T06","category":"Technical","difficulty":"easy",
     "question":"What is the difference between a stack and a queue? Give real-world examples.",
     "correct_answer":"A stack is LIFO (Last In First Out) like a pile of plates. A queue is FIFO (First In First Out) like a line of people.",
     "expected_keywords":["stack","queue","LIFO","FIFO","last in first out","first in first out","example"],"marks":5},

    {"id":"T07","category":"Technical","difficulty":"medium",
     "question":"Explain OOP concepts: encapsulation, inheritance, and polymorphism.",
     "correct_answer":"Encapsulation bundles data and methods hiding internals. Inheritance derives properties from a parent class. Polymorphism allows different classes via the same interface.",
     "expected_keywords":["encapsulation","inheritance","polymorphism","class","object","OOP","method"],"marks":10},

    {"id":"T08","category":"Technical","difficulty":"medium",
     "question":"What is SQL indexing and why is it important for performance?",
     "correct_answer":"An index is a data structure that improves data retrieval speed. Without indexes, databases do full table scans. Indexes reduce query time on large datasets.",
     "expected_keywords":["index","performance","query","speed","retrieval","table scan","data structure"],"marks":5},

    {"id":"T09","category":"Technical","difficulty":"hard",
     "question":"What are Python generators and how do they differ from regular functions?",
     "correct_answer":"Generators use yield instead of return to lazily produce values. They maintain state between calls and are memory-efficient for large datasets.",
     "expected_keywords":["generator","yield","lazy","memory","state","iterator","sequence","on demand"],"marks":10},

    {"id":"T10","category":"Technical","difficulty":"medium",
     "question":"Explain RESTful API principles and HTTP methods.",
     "correct_answer":"REST uses stateless HTTP communication. Methods: GET (retrieve), POST (create), PUT (update), DELETE (remove). Resources are identified by URLs.",
     "expected_keywords":["REST","stateless","HTTP","GET","POST","PUT","DELETE","URL","API","resource"],"marks":5},

    {"id":"T11","category":"Technical","difficulty":"easy",
     "question":"What is version control and why is Git important?",
     "correct_answer":"Version control tracks code changes over time. Git is distributed, allowing teams to collaborate, branch, merge, and revert to previous versions.",
     "expected_keywords":["version control","git","branch","merge","commit","history","collaborate","track"],"marks":5},

    {"id":"T12","category":"Technical","difficulty":"hard",
     "question":"What are design patterns? Explain Singleton and Factory patterns.",
     "correct_answer":"Design patterns are reusable solutions to common software problems. Singleton ensures one class instance. Factory creates objects without specifying concrete classes.",
     "expected_keywords":["design pattern","singleton","factory","instance","interface","reusable","class"],"marks":10},

    {"id":"T13","category":"Technical","difficulty":"medium",
     "question":"Explain Big O notation. What is the time complexity of binary search?",
     "correct_answer":"Big O describes worst-case complexity as input grows. Binary search is O(log n) because it halves the search space each step.",
     "expected_keywords":["big O","complexity","binary search","log n","O(log n)","algorithm","worst case"],"marks":5},

    {"id":"T14","category":"Technical","difficulty":"easy",
     "question":"What is the difference between synchronous and asynchronous programming?",
     "correct_answer":"Synchronous code blocks until each operation completes. Asynchronous allows other tasks while waiting for I/O using async/await patterns.",
     "expected_keywords":["synchronous","asynchronous","blocking","async","await","concurrent","I/O"],"marks":5},

    {"id":"T15","category":"Technical","difficulty":"medium",
     "question":"What is Docker and how does containerization benefit deployment?",
     "correct_answer":"Docker packages code with dependencies into containers ensuring consistent environments across dev, test, and production.",
     "expected_keywords":["docker","container","deployment","image","environment","consistent","isolation"],"marks":5},

    # ── HR (10) ──────────────────────────────────────────────────────
    {"id":"H01","category":"HR","difficulty":"easy",
     "question":"Tell me about yourself and your professional background.",
     "correct_answer":"A strong answer covers education, key experience, relevant skills, and career goals aligned with the role.",
     "expected_keywords":["experience","skills","background","education","career","professional","role","goal"],"marks":5},

    {"id":"H02","category":"HR","difficulty":"easy",
     "question":"Why do you want to work for our company?",
     "correct_answer":"A good answer references specific company values, products, culture, or mission and shows genuine research.",
     "expected_keywords":["company","values","culture","mission","growth","opportunity","align","contribute"],"marks":5},

    {"id":"H03","category":"HR","difficulty":"medium",
     "question":"Describe a time you dealt with a difficult coworker.",
     "correct_answer":"Using STAR: Describe the conflict, actions taken (communication, mediation), and positive outcome. Focus on professionalism.",
     "expected_keywords":["conflict","communication","resolve","team","professional","situation","outcome","coworker"],"marks":5},

    {"id":"H04","category":"HR","difficulty":"easy",
     "question":"What are your greatest strengths?",
     "correct_answer":"Pick 2-3 relevant strengths with specific examples: problem-solving, leadership, attention to detail, or communication.",
     "expected_keywords":["strength","problem solving","leadership","communication","skill","example","contribute"],"marks":5},

    {"id":"H05","category":"HR","difficulty":"medium",
     "question":"Where do you see yourself in 5 years?",
     "correct_answer":"Show ambition aligned with the company's growth, mention skill development, responsibility growth, and team contribution.",
     "expected_keywords":["goal","growth","career","skills","develop","responsibility","contribute","future"],"marks":5},

    {"id":"H06","category":"HR","difficulty":"medium",
     "question":"How do you handle pressure and tight deadlines?",
     "correct_answer":"Specific strategies: prioritization, time management, communicating blockers, breaking tasks into parts, staying calm.",
     "expected_keywords":["priority","deadline","pressure","manage","time","organize","communicate","calm"],"marks":5},

    {"id":"H07","category":"HR","difficulty":"easy",
     "question":"What motivates you in your work?",
     "correct_answer":"Authentic answers include learning, problem-solving, impact, teamwork, or mastery tied to the role.",
     "expected_keywords":["motivation","passion","learn","challenge","impact","team","grow","achieve"],"marks":5},

    {"id":"H08","category":"HR","difficulty":"medium",
     "question":"Describe a failure and what you learned from it.",
     "correct_answer":"Honestly describe a setback, take responsibility, explain the lesson, and show how it improved future performance.",
     "expected_keywords":["failure","learn","mistake","responsibility","improve","lesson","outcome","growth"],"marks":5},

    {"id":"H09","category":"HR","difficulty":"easy",
     "question":"What is your expected salary?",
     "correct_answer":"Research market rates, provide a range based on experience, show flexibility, frame around total compensation.",
     "expected_keywords":["salary","range","market","compensation","flexible","experience","negotiate"],"marks":5},

    {"id":"H10","category":"HR","difficulty":"medium",
     "question":"How do you prioritize tasks when managing multiple projects?",
     "correct_answer":"Assess urgency and importance (Eisenhower Matrix), use tools like Jira/Trello, communicate blockers early.",
     "expected_keywords":["prioritize","urgent","important","manage","tool","communicate","deadline","organize"],"marks":5},

    # ── APTITUDE (10) ────────────────────────────────────────────────
    {"id":"A01","category":"Aptitude","difficulty":"easy",
     "question":"If a train travels 300 km in 3 hours, what is its speed? How long for 500 km?",
     "correct_answer":"Speed = 300/3 = 100 km/h. Time for 500 km = 500/100 = 5 hours.",
     "expected_keywords":["100","speed","5 hours","km/h","average","300","500"],"marks":5},

    {"id":"A02","category":"Aptitude","difficulty":"medium",
     "question":"What comes next in the series: 2, 6, 12, 20, 30, ?",
     "correct_answer":"Differences are 4, 6, 8, 10 — increasing by 2. Next difference = 12. Answer = 42.",
     "expected_keywords":["42","pattern","difference","sequence","12","increasing"],"marks":5},

    {"id":"A03","category":"Aptitude","difficulty":"easy",
     "question":"If 15 workers complete a job in 12 days, how many days will 20 workers take?",
     "correct_answer":"15 × 12 = 20 × d → d = 180/20 = 9 days.",
     "expected_keywords":["9","9 days","inverse","proportion","workers","180"],"marks":5},

    {"id":"A04","category":"Aptitude","difficulty":"medium",
     "question":"A product is sold at 20% profit. Cost price is ₹500. What is the selling price?",
     "correct_answer":"Selling price = 500 × 1.20 = ₹600.",
     "expected_keywords":["600","₹600","profit","selling price","20%","500","1.2"],"marks":5},

    {"id":"A05","category":"Aptitude","difficulty":"hard",
     "question":"If log₂(8) = x, what is x? What is log₁₀(1000)?",
     "correct_answer":"log₂(8) = 3 because 2³ = 8. log₁₀(1000) = 3 because 10³ = 1000.",
     "expected_keywords":["3","log","logarithm","base","exponent","2^3","10^3"],"marks":10},

    {"id":"A06","category":"Aptitude","difficulty":"easy",
     "question":"What is 15% of 240?",
     "correct_answer":"15% of 240 = (15/100) × 240 = 36.",
     "expected_keywords":["36","15%","240","percent"],"marks":5},

    {"id":"A07","category":"Aptitude","difficulty":"medium",
     "question":"Two pipes fill a tank in 6 and 9 hours. How long to fill together?",
     "correct_answer":"Combined rate = 1/6 + 1/9 = 5/18. Time = 18/5 = 3.6 hours.",
     "expected_keywords":["3.6","18/5","combined","rate","together"],"marks":5},

    {"id":"A08","category":"Aptitude","difficulty":"easy",
     "question":"If today is Monday, what day will it be after 100 days?",
     "correct_answer":"100 ÷ 7 = 14 weeks + 2 days. Monday + 2 = Wednesday.",
     "expected_keywords":["wednesday","100","7","remainder","2","days"],"marks":5},

    {"id":"A09","category":"Aptitude","difficulty":"medium",
     "question":"A car depreciates 10% each year. Value after 2 years from ₹1,00,000?",
     "correct_answer":"Year 1: 1,00,000 × 0.9 = ₹90,000. Year 2: 90,000 × 0.9 = ₹81,000.",
     "expected_keywords":["81000","₹81,000","depreciate","0.9","90000","10%"],"marks":5},

    {"id":"A10","category":"Aptitude","difficulty":"hard",
     "question":"How many ways can 4 people be arranged in 4 seats? Choosing 2 from 5?",
     "correct_answer":"4 people in 4 seats = 4! = 24. Choosing 2 from 5 = 5C2 = 10.",
     "expected_keywords":["24","4!","factorial","10","5C2","combinations","permutation"],"marks":10},

    # ── BEHAVIORAL (10) ──────────────────────────────────────────────
    {"id":"B01","category":"Behavioral","difficulty":"medium",
     "question":"Describe a situation where you showed leadership.",
     "correct_answer":"STAR: Specific leadership moment, actions (delegating, motivating, deciding), challenges, measurable positive outcome.",
     "expected_keywords":["leadership","team","initiative","decision","outcome","delegate","motivate","result"],"marks":5},

    {"id":"B02","category":"Behavioral","difficulty":"medium",
     "question":"Tell me about working effectively in a team to solve a complex problem.",
     "correct_answer":"Describe team composition, your role, collaboration methods, how disagreements were resolved, successful outcome.",
     "expected_keywords":["team","collaborate","role","communication","problem","solve","contribute","outcome"],"marks":5},

    {"id":"B03","category":"Behavioral","difficulty":"easy",
     "question":"How do you approach learning new technologies or skills?",
     "correct_answer":"Structured learning: online courses, documentation, projects, community, mentorship. Shows curiosity and adaptability.",
     "expected_keywords":["learn","course","practice","documentation","project","community","adapt","skill"],"marks":5},

    {"id":"B04","category":"Behavioral","difficulty":"hard",
     "question":"Describe making a difficult decision with limited information.",
     "correct_answer":"Gather available data, assess risks, consult others, decide with reasoning, reflect on outcome. Shows analytical thinking.",
     "expected_keywords":["decision","risk","uncertainty","data","analyze","consult","outcome","reflect"],"marks":10},

    {"id":"B05","category":"Behavioral","difficulty":"medium",
     "question":"How do you handle receiving negative feedback or criticism?",
     "correct_answer":"Open to feedback, separate emotion from message, ask clarifying questions, create action plan, follow up.",
     "expected_keywords":["feedback","criticism","improve","open","listen","action","grow","respond"],"marks":5},

    {"id":"B06","category":"Behavioral","difficulty":"medium",
     "question":"Tell me about a time you went above and beyond in your work.",
     "correct_answer":"Specific initiative beyond job requirements, reasoning, actions, and tangible impact on team or product.",
     "expected_keywords":["initiative","extra","beyond","impact","ownership","dedicate","contribute","result"],"marks":5},

    {"id":"B07","category":"Behavioral","difficulty":"easy",
     "question":"How do you manage work-life balance during high-pressure periods?",
     "correct_answer":"Healthy boundaries, time-blocking, self-care, communicating availability, recovery strategies.",
     "expected_keywords":["balance","boundary","priority","health","time","communicate","rest","manage"],"marks":5},

    {"id":"B08","category":"Behavioral","difficulty":"medium",
     "question":"Describe a time you disagreed with your manager. How did you handle it?",
     "correct_answer":"Respectful disagreement, presenting facts/data, listening to their view, reaching consensus or accepting gracefully.",
     "expected_keywords":["disagree","manager","respectful","communicate","data","listen","consensus"],"marks":5},

    {"id":"B09","category":"Behavioral","difficulty":"easy",
     "question":"What do you do when stuck on a problem for a long time?",
     "correct_answer":"Take a break, research, break into parts, ask colleagues, consult documentation, reflect on root cause.",
     "expected_keywords":["break","research","ask","colleague","documentation","debug","approach","help"],"marks":5},

    {"id":"B10","category":"Behavioral","difficulty":"hard",
     "question":"Tell me about your most significant professional achievement.",
     "correct_answer":"Specific metrics (e.g., 'reduced load time by 40%'), business impact, personal contribution, lessons learned.",
     "expected_keywords":["achievement","impact","metric","result","contribution","learn","significant"],"marks":10},
]

ROUNDS = ["Technical", "HR", "Aptitude", "Behavioral"]
QUESTIONS_PER_ROUND = 5
print(f"✅ Loaded {len(QUESTION_BANK)} questions across {len(ROUNDS)} rounds")


# ══════════════════════════════════════════════════════════════════════
# ── CELL 4 ── NLP ENGINE
# ══════════════════════════════════════════════════════════════════════

def load_questions(category, count=5):
    pool = [q for q in QUESTION_BANK if q["category"] == category]
    return random.sample(pool, min(count, len(pool)))

def extract_keywords(text):
    if SPACY_AVAILABLE and nlp:
        doc = nlp(text.lower())
        return [t.lemma_ for t in doc if not t.is_stop and not t.is_punct and t.is_alpha and len(t.text) > 2]
    stopwords = {"the","a","an","is","are","was","be","been","it","to","of","and","or","in","on","at","for"}
    words = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())
    return [w for w in words if w not in stopwords]

def semantic_similarity(text1, text2):
    if SPACY_AVAILABLE and nlp:
        try:
            d1, d2 = nlp(text1[:500]), nlp(text2[:500])
            if d1.vector_norm and d2.vector_norm:
                return d1.similarity(d2)
        except: pass
    w1, w2 = set(extract_keywords(text1)), set(extract_keywords(text2))
    if not w1 or not w2: return 0.0
    return len(w1 & w2) / len(w1 | w2)

def keyword_match_score(answer, keywords):
    if not keywords: return 0.5
    ans = answer.lower()
    return sum(1 for k in keywords if k.lower() in ans) / len(keywords)

def sentiment_score(text):
    pos = {"good","great","excellent","positive","strong","successful","improve","efficient","confident","achieve"}
    neg = {"bad","poor","fail","struggle","weak","issue","problem","difficult","never","wrong"}
    t = text.lower()
    p, n = sum(1 for w in pos if w in t), sum(1 for w in neg if w in t)
    return "positive" if p > n else ("negative" if n > p else "neutral")

def evaluate_answer(user_answer, question):
    if not user_answer or not user_answer.strip():
        return {"score":0,"max_marks":question["marks"],"grade":"No Answer",
                "keyword_score":0.0,"similarity_score":0.0,"feedback":"⚠️ No answer provided.","sentiment":"neutral","word_count":0}
    mm = question["marks"]
    kw_score  = keyword_match_score(user_answer, question.get("expected_keywords",[]))
    sim_score = semantic_similarity(user_answer, question.get("correct_answer",""))
    wc        = len(user_answer.split())
    too_short = wc < 10
    composite = min((kw_score*0.5) + (sim_score*0.4) + (0.1 if not too_short else 0.0), 1.0)

    if composite >= 0.65:
        awarded, grade, fb = mm,    "✅ Excellent",          "Great answer! You covered the key concepts well."
    elif composite >= 0.35:
        awarded, grade, fb = mm/2,  "🟡 Partial Credit",     f"Partial credit. Some keywords missed: {', '.join(question.get('expected_keywords',[])[:3])}."
    else:
        awarded, grade, fb = 0,     "❌ Needs Improvement",  f"Answer needs more depth. Key topics: {', '.join(question.get('expected_keywords',[])[:3])}."

    if too_short and awarded > 0:
        awarded = max(0, awarded * 0.7)
        fb += " Answer is too brief — please elaborate."

    return {"score":round(awarded,1),"max_marks":mm,"grade":grade,
            "keyword_score":round(kw_score,2),"similarity_score":round(sim_score,2),
            "composite":round(composite,2),"feedback":fb,
            "sentiment":sentiment_score(user_answer),"word_count":wc}

print("✅ NLP Evaluation Engine ready")


# ══════════════════════════════════════════════════════════════════════
# ── CELL 5 ── FEEDBACK GENERATOR & DATA STORAGE
# ══════════════════════════════════════════════════════════════════════

def generate_feedback(candidate_info, scores, category_scores):
    total  = scores.get("total", 0)
    max_t  = scores.get("max_total", 1)
    pct    = (total / max_t * 100) if max_t else 0
    name   = candidate_info.get("name", "Candidate")
    pos    = candidate_info.get("position", "the applied role")

    if pct >= 75:
        rec = "🟢 **STRONGLY RECOMMENDED** for hire"
        rec_d = f"{name} has demonstrated exceptional competency — strong fit for {pos}."
    elif pct >= 55:
        rec = "🟡 **RECOMMENDED** with reservations"
        rec_d = f"{name} shows adequate skills but would benefit from further development."
    elif pct >= 35:
        rec = "🟠 **BORDERLINE** — Further evaluation suggested"
        rec_d = f"{name} needs improvement before being considered for {pos}."
    else:
        rec = "🔴 **NOT RECOMMENDED** at this time"
        rec_d = f"{name} does not currently meet minimum requirements for {pos}."

    insights, strengths, weaknesses = [], [], []
    for cat, data in category_scores.items():
        cp = (data["scored"] / data["max"] * 100) if data["max"] else 0
        if cp >= 70:
            insights.append(f"• **{cat}**: 💪 Strong ({cp:.0f}%)")
            strengths.append(f"• Strong {cat} knowledge")
        elif cp >= 40:
            insights.append(f"• **{cat}**: 📈 Moderate ({cp:.0f}%)")
        else:
            insights.append(f"• **{cat}**: ⚠️ Weak ({cp:.0f}%)")
            weaknesses.append(f"• {cat} skills need development")

    return f"""## 📋 Interview Evaluation Report

**Candidate:** {name} | **Position:** {pos}
**Date:** {datetime.now().strftime('%B %d, %Y %H:%M')} | **Score:** {total:.1f}/{max_t} ({pct:.1f}%)

---
## 🏆 Overall Recommendation
{rec}
{rec_d}

---
## 📊 Category Performance
{''.join(insights) if insights else '• No data'}

---
## 💡 Key Strengths
{''.join(strengths) if strengths else '• No standout strengths identified'}

## 🎯 Areas for Development
{''.join(weaknesses) if weaknesses else '• No major weaknesses — well-rounded candidate'}

---
## 📌 Next Steps
{"✅ Proceed to final round interview and reference checks." if pct >= 55 else "📚 Strengthen technical and communication skills before reapplying."}
""".strip()

def save_results(session):
    records = []
    if os.path.exists(RESULTS_FILE):
        try:
            with open(RESULTS_FILE) as f: records = json.load(f)
        except: records = []
    records.append({
        "id": f"INT_{int(time.time())}",
        "timestamp": datetime.now().isoformat(),
        "candidate": session.get("candidate_info", {}),
        "scores": session.get("scores", {}),
        "category_scores": session.get("category_scores", {}),
        "feedback": session.get("feedback", ""),
    })
    with open(RESULTS_FILE, "w") as f: json.dump(records, f, indent=2)

def load_all_results():
    if not os.path.exists(RESULTS_FILE): return []
    try:
        with open(RESULTS_FILE) as f: return json.load(f)
    except: return []

print("✅ Feedback generator & storage ready")


# ══════════════════════════════════════════════════════════════════════
# ── CELL 6 ── SESSION & INTERVIEW FLOW
# ══════════════════════════════════════════════════════════════════════

def make_fresh_session():
    return {
        "stage": "welcome",
        "candidate_info": {},
        "current_round_index": 0,
        "current_question_index": 0,
        "round_questions": [],
        "scores": {"total": 0, "max_total": 0},
        "category_scores": {r: {"scored":0,"max":0,"count":0} for r in ROUNDS},
        "correct_count": 0,
        "incorrect_count": 0,
        "questions_answered": [],
        "chat_history": [],
        "feedback": "",
    }

def total_questions(): return len(ROUNDS) * QUESTIONS_PER_ROUND

def _format_question(q, qnum, total, round_name):
    badges = {"easy":"🟢 Easy","medium":"🟡 Medium","hard":"🔴 Hard"}
    return (f"**Round: {round_name} | Question {qnum} of {total}**\n\n"
            f"**{q['question']}**\n\n"
            f"_{badges.get(q['difficulty'],'⚪')} | {q['marks']} marks_")

def get_score_display(session):
    s    = session.get("scores", {})
    tot  = s.get("total", 0)
    mx   = s.get("max_total", 0)
    pct  = (tot/mx*100) if mx else 0
    ri   = session.get("current_round_index", 0)
    qi   = session.get("current_question_index", 0)
    rn   = ROUNDS[ri] if ri < len(ROUNDS) else "Complete"
    qn   = min(ri*QUESTIONS_PER_ROUND + qi + 1, total_questions())
    return (f"**📊 Live Score**\n\n"
            f"🏆 **{tot:.1f} / {mx}** ({pct:.1f}%)\n\n"
            f"✅ {session.get('correct_count',0)} correct | ❌ {session.get('incorrect_count',0)} incorrect\n\n"
            f"📍 **{rn} Round** | Q{qn}/{total_questions()}")

def get_progress_pct(session):
    if session.get("stage") == "results": return 100.0
    ri = session.get("current_round_index", 0)
    qi = session.get("current_question_index", 0)
    return min(((ri*QUESTIONS_PER_ROUND + qi) / total_questions()) * 100, 99.9)

def begin_interview(session):
    session.update(stage="interview", current_round_index=0, current_question_index=0)
    session["round_questions"] = load_questions(ROUNDS[0], QUESTIONS_PER_ROUND)
    q = session["round_questions"][0]
    hist = session.get("chat_history", [])
    hist.append(("", _format_question(q, 1, total_questions(), ROUNDS[0])))
    session["chat_history"] = hist
    return session

def submit_answer(user_msg, session):
    if session.get("stage") != "interview" or not user_msg.strip():
        return session, session.get("chat_history", [])
    ri, qi = session["current_round_index"], session["current_question_index"]
    q      = session["round_questions"][qi]
    ev     = evaluate_answer(user_msg, q)
    hist   = session.get("chat_history", [])

    # Update scores
    session["scores"]["total"]   = round(session["scores"]["total"] + ev["score"], 1)
    session["scores"]["max_total"] += q["marks"]
    cat = q["category"]
    session["category_scores"][cat]["scored"] = round(session["category_scores"][cat]["scored"] + ev["score"], 1)
    session["category_scores"][cat]["max"]    += q["marks"]
    session["category_scores"][cat]["count"]  += 1
    if ev["score"] >= q["marks"] * 0.65: session["correct_count"]   += 1
    else:                                session["incorrect_count"] += 1

    session["questions_answered"].append({
        "q": q["question"], "a": user_msg,
        "score": ev["score"], "max": q["marks"], "grade": ev["grade"]
    })

    fb = (f"{ev['grade']} | **{ev['score']:.1f}/{q['marks']} marks**\n\n"
          f"{ev['feedback']}\n\n"
          f"_Keyword match: {ev['keyword_score']*100:.0f}% | Relevance: {ev['similarity_score']*100:.0f}%_")
    hist.append((user_msg, fb))

    # Advance state
    next_qi = qi + 1
    if next_qi < len(session["round_questions"]):
        session["current_question_index"] = next_qi
        qnum = ri*QUESTIONS_PER_ROUND + next_qi + 1
        hist.append(("", _format_question(session["round_questions"][next_qi], qnum, total_questions(), ROUNDS[ri])))
    else:
        next_ri = ri + 1
        if next_ri < len(ROUNDS):
            session["current_round_index"] = next_ri
            session["current_question_index"] = 0
            nqs = load_questions(ROUNDS[next_ri], QUESTIONS_PER_ROUND)
            session["round_questions"] = nqs
            cat_d = session["category_scores"][ROUNDS[ri]]
            cp = (cat_d["scored"]/cat_d["max"]*100) if cat_d["max"] else 0
            hist.append(("", (f"🎉 **{ROUNDS[ri]} Round Complete!**\n\n"
                              f"Score: **{cat_d['scored']:.1f}/{cat_d['max']}** ({cp:.0f}%)\n\n"
                              f"---\n\n**Next: {ROUNDS[next_ri]} Round**\n\n"
                              + _format_question(nqs[0], next_ri*QUESTIONS_PER_ROUND+1, total_questions(), ROUNDS[next_ri]))))
        else:
            session["stage"] = "results"
            session["feedback"] = generate_feedback(session["candidate_info"], session["scores"], session["category_scores"])
            save_results(session)
            s = session["scores"]
            pct = (s["total"]/s["max_total"]*100) if s["max_total"] else 0
            hist.append(("", (f"🏁 **Interview Complete!**\n\n"
                              f"**Final Score: {s['total']:.1f}/{s['max_total']} ({pct:.1f}%)**\n\n"
                              f"✅ Correct: {session['correct_count']} | ❌ Incorrect: {session['incorrect_count']}\n\n"
                              "👇 Scroll down to see your full results dashboard.")))

    session["chat_history"] = hist
    return session, hist

print("✅ Interview flow controller ready")


# ══════════════════════════════════════════════════════════════════════
# ── CELL 7 ── ADMIN HELPERS
# ══════════════════════════════════════════════════════════════════════

def get_admin_summary():
    records = load_all_results()
    if not records: return "📭 No interview records yet. Complete an interview to see data here."
    avg = sum((r["scores"].get("total",0)/r["scores"].get("max_total",1)*100) for r in records) / len(records)
    lines = [f"## 📊 Admin Dashboard\n**Total Interviews:** {len(records)} | **Avg Score:** {avg:.1f}%\n",
             "| # | Name | Position | Score | % | Date |",
             "|---|------|----------|-------|---|------|"]
    for i, r in enumerate(reversed(records[-20:]), 1):
        c, s = r.get("candidate",{}), r.get("scores",{})
        p = (s.get("total",0)/s.get("max_total",1)*100) if s.get("max_total") else 0
        lines.append(f"| {i} | {c.get('name','N/A')} | {c.get('position','N/A')} | {s.get('total',0):.1f}/{s.get('max_total',0)} | {p:.1f}% | {r.get('timestamp','')[:10]} |")
    return "\n".join(lines)

def search_candidate(query):
    if not query.strip(): return "Enter a name or position to search."
    ql = query.lower()
    found = [r for r in load_all_results()
             if ql in r.get("candidate",{}).get("name","").lower()
             or ql in r.get("candidate",{}).get("position","").lower()]
    if not found: return f"No records found for '{query}'."
    out = [f"### 🔍 Results for '{query}'\n"]
    for r in found:
        c, s = r.get("candidate",{}), r.get("scores",{})
        p = (s.get("total",0)/s.get("max_total",1)*100) if s.get("max_total") else 0
        out.append(f"**{c.get('name')}** | {c.get('position')} | Score: {p:.1f}% | {r.get('timestamp','')[:10]}")
    return "\n\n".join(out)

print("✅ Admin dashboard helpers ready")


# ══════════════════════════════════════════════════════════════════════
# ── CELL 8 ── CUSTOM CSS (Dark Glassmorphism)
# ══════════════════════════════════════════════════════════════════════

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Sora:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

*, *::before, *::after { box-sizing: border-box; }

body, .gradio-container {
    font-family: 'Sora', sans-serif !important;
    background: #080c14 !important;
    color: #e2e8f0 !important;
}

.gradio-container::before {
    content: '';
    position: fixed; top: 0; left: 0; right: 0; bottom: 0;
    background-image:
        linear-gradient(rgba(56,189,248,0.025) 1px, transparent 1px),
        linear-gradient(90deg, rgba(56,189,248,0.025) 1px, transparent 1px);
    background-size: 48px 48px;
    pointer-events: none; z-index: 0;
}

.hero-header {
    background: linear-gradient(135deg, #0f1728 0%, #0d1f3c 50%, #0f1728 100%);
    border-bottom: 1px solid rgba(56,189,248,0.15);
    padding: 28px 40px; text-align: center; position: relative; overflow: hidden;
}
.hero-header::after {
    content: ''; position: absolute; top: -60%; left: 20%; width: 60%; height: 200%;
    background: radial-gradient(ellipse, rgba(56,189,248,0.07) 0%, transparent 60%);
    pointer-events: none;
}
.hero-title {
    font-size: clamp(1.6rem, 4vw, 2.2rem) !important; font-weight: 700 !important;
    background: linear-gradient(135deg, #38bdf8, #818cf8, #f472b6) !important;
    -webkit-background-clip: text !important; -webkit-text-fill-color: transparent !important;
    background-clip: text !important; letter-spacing: -0.5px; margin-bottom: 6px !important;
}
.hero-sub {
    color: #64748b !important; font-size: 0.85rem !important;
    letter-spacing: 2px; text-transform: uppercase;
}

.glass-card {
    background: rgba(15,23,42,0.75) !important;
    border: 1px solid rgba(56,189,248,0.12) !important;
    border-radius: 16px !important; backdrop-filter: blur(20px) !important;
    padding: 24px !important;
    box-shadow: 0 8px 32px rgba(0,0,0,0.4), inset 0 1px 0 rgba(255,255,255,0.04) !important;
    transition: border-color 0.3s !important;
}
.glass-card:hover { border-color: rgba(56,189,248,0.25) !important; }

.score-panel {
    background: linear-gradient(135deg, rgba(15,23,42,0.95), rgba(13,31,60,0.95)) !important;
    border: 1px solid rgba(56,189,248,0.2) !important; border-radius: 16px !important;
    padding: 20px !important;
    box-shadow: 0 0 30px rgba(56,189,248,0.04), 0 8px 32px rgba(0,0,0,0.5) !important;
}

input[type="text"], input[type="email"], textarea, .gr-textbox textarea {
    background: rgba(15,23,42,0.8) !important;
    border: 1px solid rgba(56,189,248,0.15) !important;
    border-radius: 10px !important; color: #e2e8f0 !important;
    font-family: 'Sora', sans-serif !important; font-size: 0.9rem !important;
    padding: 10px 14px !important; transition: border-color 0.2s, box-shadow 0.2s !important;
}
input:focus, textarea:focus {
    border-color: rgba(56,189,248,0.5) !important;
    box-shadow: 0 0 0 3px rgba(56,189,248,0.08) !important; outline: none !important;
}

button.btn-primary, .btn-primary {
    background: linear-gradient(135deg, #0ea5e9, #6366f1) !important;
    border: none !important; border-radius: 10px !important; color: white !important;
    font-family: 'Sora', sans-serif !important; font-weight: 600 !important;
    font-size: 0.9rem !important; padding: 12px 28px !important; cursor: pointer !important;
    transition: opacity 0.2s, transform 0.15s, box-shadow 0.2s !important;
    box-shadow: 0 4px 20px rgba(14,165,233,0.3) !important;
}
button.btn-primary:hover { opacity: 0.9 !important; transform: translateY(-1px) !important; }

button.btn-secondary, .btn-secondary {
    background: rgba(30,41,59,0.8) !important;
    border: 1px solid rgba(56,189,248,0.2) !important; border-radius: 10px !important;
    color: #94a3b8 !important; font-family: 'Sora', sans-serif !important;
    font-size: 0.85rem !important; padding: 10px 20px !important;
    transition: all 0.2s !important;
}
button.btn-secondary:hover { background: rgba(30,41,59,1) !important; color: #e2e8f0 !important; }

.gr-chatbot, .chatbot {
    background: rgba(8,12,20,0.6) !important;
    border: 1px solid rgba(56,189,248,0.1) !important;
    border-radius: 16px !important; font-family: 'Sora', sans-serif !important;
}

.tab-nav button {
    background: transparent !important; border: none !important;
    border-bottom: 2px solid transparent !important; color: #64748b !important;
    font-family: 'Sora', sans-serif !important; font-weight: 500 !important;
    padding: 10px 20px !important; transition: all 0.2s !important;
}
.tab-nav button.selected, .tab-nav button[aria-selected="true"] {
    color: #38bdf8 !important; border-bottom-color: #38bdf8 !important;
}

.progress-bar { height: 6px; background: rgba(30,41,59,0.8); border-radius: 99px; overflow: hidden; margin: 8px 0; }
.progress-fill { height: 100%; background: linear-gradient(90deg, #38bdf8, #818cf8); border-radius: 99px; transition: width 0.5s ease; }

.gr-markdown h1, .gr-markdown h2 { color: #38bdf8 !important; }
.gr-markdown h3 { color: #818cf8 !important; }
.gr-markdown table { width: 100%; border-collapse: collapse; }
.gr-markdown th { background: rgba(56,189,248,0.1) !important; color: #38bdf8 !important; padding: 8px 12px !important; font-size: 0.82rem !important; }
.gr-markdown td { padding: 8px 12px !important; border-bottom: 1px solid rgba(255,255,255,0.05) !important; font-size: 0.85rem !important; color: #cbd5e1 !important; }
.gr-markdown code { background: rgba(56,189,248,0.08) !important; color: #38bdf8 !important; font-family: 'JetBrains Mono', monospace !important; border-radius: 4px !important; padding: 2px 6px !important; }

label, .gr-form label { color: #94a3b8 !important; font-size: 0.82rem !important; font-weight: 500 !important; text-transform: uppercase !important; letter-spacing: 0.3px !important; }

::-webkit-scrollbar { width: 5px; }
::-webkit-scrollbar-track { background: rgba(15,23,42,0.5); }
::-webkit-scrollbar-thumb { background: rgba(56,189,248,0.3); border-radius: 99px; }

.error-text { color: #f87171 !important; font-size: 0.85rem !important; }
.full-width { width: 100% !important; }
"""

print("✅ CSS styles loaded")


# ══════════════════════════════════════════════════════════════════════
# ── CELL 9 ── BUILD GRADIO APP
# ══════════════════════════════════════════════════════════════════════

def build_app():
    with gr.Blocks(css=CSS, title="AI Interview Assistant", theme=gr.themes.Base()) as demo:

        session_state = gr.State(make_fresh_session())

        # ── HEADER ────────────────────────────────────────────────
        gr.HTML("""
        <div class="hero-header">
            <div class="hero-title">🤖 AI Interview Assistant</div>
            <div class="hero-sub">Intelligent · Real-time · Multi-round Evaluation Platform</div>
        </div>
        """)

        with gr.Tabs():

            # ── TAB 1: INTERVIEW ─────────────────────────────────
            with gr.Tab("🎤 Interview"):
                with gr.Row(equal_height=False):

                    # LEFT: Chat area
                    with gr.Column(scale=7):

                        # WELCOME PANEL
                        with gr.Column(visible=True, elem_classes="glass-card") as welcome_panel:
                            gr.Markdown("""
## 👋 Welcome to AI Interview Assistant

Your intelligent virtual interviewer, powered by NLP evaluation.

**Session overview:**
- 🔵 **Technical Round** — 5 questions (Python, SQL, concepts)
- 🟢 **HR Round** — 5 questions (communication & culture fit)
- 🟡 **Aptitude Round** — 5 questions (logic & math)
- 🟣 **Behavioral Round** — 5 questions (soft skills & STAR)

**📊 20 questions total** | Real-time scoring | AI feedback report

> **Tips:** Give detailed answers (3–5 sentences). Use technical terminology. Think before answering.
                            """)
                            start_reg_btn = gr.Button("🚀 Start Registration →", elem_classes="btn-primary full-width")

                        # REGISTRATION FORM
                        with gr.Column(visible=False, elem_classes="glass-card") as reg_panel:
                            gr.Markdown("### 📋 Candidate Registration")
                            with gr.Row():
                                reg_name  = gr.Textbox(label="Full Name *", placeholder="e.g. Priya Sharma")
                                reg_email = gr.Textbox(label="Email Address *", placeholder="priya@example.com")
                            with gr.Row():
                                reg_phone = gr.Textbox(label="Phone Number", placeholder="+91 98765 43210")
                                reg_edu   = gr.Textbox(label="Education", placeholder="B.Tech Computer Science")
                            with gr.Row():
                                reg_skills = gr.Textbox(label="Key Skills", placeholder="Python, SQL, Machine Learning")
                                reg_exp    = gr.Textbox(label="Years of Experience", placeholder="3")
                            reg_pos    = gr.Textbox(label="Position Applied For *", placeholder="Senior Software Engineer")
                            reg_error  = gr.Markdown("", elem_classes="error-text")
                            reg_submit_btn = gr.Button("✅ Complete Registration", elem_classes="btn-primary full-width")

                        # CHAT INTERVIEW PANEL
                        with gr.Column(visible=False) as interview_panel:
                            progress_html = gr.HTML("""
                            <div class="progress-bar">
                              <div class="progress-fill" style="width:0%"></div>
                            </div>
                            <div style="text-align:right;font-size:0.78rem;color:#64748b;margin-top:4px">0% Complete</div>
                            """)
                            chatbot = gr.Chatbot(
                                label="Interview",
                                height=440,
                                show_label=False,
                                bubble_full_width=False,
                                avatar_images=(None, "🤖"),
                            )
                            with gr.Row():
                                answer_input = gr.Textbox(
                                    placeholder="Type your answer here and press Enter or click Submit...",
                                    label="", lines=3, scale=5, show_label=False,
                                )
                                submit_btn = gr.Button("Submit →", scale=1, elem_classes="btn-primary")
                            begin_btn = gr.Button("▶  Begin Interview", elem_classes="btn-primary full-width", visible=False)

                    # RIGHT: Score panel
                    with gr.Column(scale=3):
                        with gr.Column(elem_classes="score-panel"):
                            gr.Markdown("### 📊 Live Score")
                            score_display = gr.Markdown("_Start the interview to see your live score_")
                            gr.HTML("<div style='height:1px;background:rgba(56,189,248,0.1);margin:12px 0'></div>")
                            gr.Markdown("**Round Tracker**")
                            gr.HTML("""
                            <div style="display:flex;flex-direction:column;gap:10px;margin-top:8px;font-size:0.83rem">
                                <div style="display:flex;justify-content:space-between">
                                    <span style="color:#38bdf8">🔵 Technical</span><span style="color:#475569">5 questions</span>
                                </div>
                                <div style="display:flex;justify-content:space-between">
                                    <span style="color:#34d399">🟢 HR</span><span style="color:#475569">5 questions</span>
                                </div>
                                <div style="display:flex;justify-content:space-between">
                                    <span style="color:#fbbf24">🟡 Aptitude</span><span style="color:#475569">5 questions</span>
                                </div>
                                <div style="display:flex;justify-content:space-between">
                                    <span style="color:#a78bfa">🟣 Behavioral</span><span style="color:#475569">5 questions</span>
                                </div>
                            </div>
                            """)

                        gr.HTML("<div style='height:16px'></div>")
                        with gr.Column(elem_classes="glass-card"):
                            gr.Markdown("### ℹ️ How Scoring Works")
                            gr.Markdown(
                                "**Full marks** → Score ≥ 65%\n\n"
                                "**Half marks** → Score 35–64%\n\n"
                                "**Zero** → Score < 35%\n\n"
                                "NLP checks keyword match + semantic relevance + answer length"
                            )

                # RESULTS DASHBOARD
                with gr.Column(visible=False, elem_classes="glass-card") as results_panel:
                    gr.Markdown("## 🏆 Interview Results Dashboard")
                    with gr.Row():
                        results_score_md    = gr.Markdown()
                        results_feedback_md = gr.Markdown()
                    restart_btn = gr.Button("🔄 Start New Interview", elem_classes="btn-secondary")

            # ── TAB 2: ADMIN DASHBOARD ───────────────────────────
            with gr.Tab("🛡️ Admin Dashboard"):
                with gr.Row():
                    refresh_btn      = gr.Button("🔄 Refresh", elem_classes="btn-secondary")
                    export_json_btn  = gr.Button("⬇️ Export JSON", elem_classes="btn-secondary")

                admin_md = gr.Markdown(get_admin_summary())

                gr.HTML("<div style='height:20px'></div>")
                gr.Markdown("### 🔍 Search Candidate")
                with gr.Row():
                    search_input = gr.Textbox(placeholder="Search by name or position...", label="", scale=4)
                    search_btn   = gr.Button("Search", scale=1, elem_classes="btn-primary")
                search_results = gr.Markdown()

                gr.HTML("<div style='height:20px'></div>")
                gr.Markdown("### 📄 Raw JSON Data")
                json_box = gr.Code(language="json", label="Interview Records", lines=18)

        # ══════════════════════════════════════════════════════════
        # EVENT WIRING
        # ══════════════════════════════════════════════════════════

        # Welcome → Registration
        def show_reg(s):
            history = [("", "👋 **Welcome to AI Interview Assistant!**\n\n"
                            "I'll be your virtual interviewer today.\n\n"
                            "📋 4 rounds | 20 questions | Real-time NLP scoring\n\n"
                            "Please complete the registration form below, then click **Complete Registration**.")]
            s["chat_history"] = history
            return s

        start_reg_btn.click(
            show_reg, inputs=[session_state], outputs=[session_state]
        ).then(
            lambda: (gr.update(visible=False), gr.update(visible=True), gr.update(visible=True)),
            outputs=[welcome_panel, reg_panel, interview_panel]
        )

        # Registration submit
        def on_register(name, email, phone, edu, skills, exp, pos, s):
            errs = []
            if not name or len(name.strip()) < 2:   errs.append("• Name must be at least 2 characters")
            if not email or "@" not in email:        errs.append("• Valid email is required")
            if not pos or len(pos.strip()) < 2:     errs.append("• Position applied for is required")
            if errs:
                return s, "\n".join(errs), gr.update(visible=True), gr.update(visible=False), [], gr.update(visible=True)

            s["candidate_info"] = {
                "name": name.strip(), "email": email.strip(), "phone": phone.strip(),
                "education": edu.strip(), "skills": skills.strip(),
                "experience": exp.strip(), "position": pos.strip(),
            }
            s["stage"] = "instructions"
            hist = s.get("chat_history", [])
            hist.append(("", f"✅ **Registration Complete!**\n\nWelcome, **{name}**!\n\n"
                            f"📋 Position: **{pos}** | 🎓 Education: {edu or 'N/A'} | 💼 Experience: {exp or 'N/A'} yrs\n\n"
                            "Click **▶ Begin Interview** when you're ready!"))
            s["chat_history"] = hist
            return s, "", gr.update(visible=False), gr.update(visible=True), hist, gr.update(visible=True)

        reg_submit_btn.click(
            on_register,
            inputs=[reg_name, reg_email, reg_phone, reg_edu, reg_skills, reg_exp, reg_pos, session_state],
            outputs=[session_state, reg_error, reg_panel, begin_btn, chatbot, interview_panel]
        )

        # Begin interview
        def on_begin(s):
            s = begin_interview(s)
            return s, s.get("chat_history",[]), get_score_display(s), gr.update(visible=False)

        begin_btn.click(
            on_begin,
            inputs=[session_state],
            outputs=[session_state, chatbot, score_display, begin_btn]
        )

        # Submit answer
        def on_submit(user_msg, s):
            if not user_msg or not user_msg.strip():
                return s, s.get("chat_history",[]), "", get_score_display(s), \
                       gr.update(), gr.update(visible=False), "", ""

            s, hist = submit_answer(user_msg, s)
            pct = get_progress_value_local(s)
            prog = (f'<div class="progress-bar"><div class="progress-fill" style="width:{pct:.1f}%"></div></div>'
                    f'<div style="text-align:right;font-size:0.78rem;color:#64748b;margin-top:4px">{pct:.0f}% Complete</div>')

            show_results = s.get("stage") == "results"
            rscore, rfb  = "", ""
            if show_results:
                sc  = s["scores"]
                tot, mx = sc.get("total",0), sc.get("max_total",1)
                p   = (tot/mx*100) if mx else 0
                rscore = (f"## 🏆 Final Score: {tot:.1f}/{mx} ({p:.1f}%)\n\n"
                          f"✅ **Correct:** {s.get('correct_count',0)} | ❌ **Incorrect:** {s.get('incorrect_count',0)}\n\n"
                          "### Category Breakdown\n\n"
                          + "\n".join([
                              f"- **{c}**: {d['scored']:.1f}/{d['max']} ({d['scored']/d['max']*100:.0f}%)" if d['max'] else f"- **{c}**: N/A"
                              for c, d in s.get("category_scores",{}).items()
                          ]))
                rfb = s.get("feedback", "")

            return (s, hist, "", get_score_display(s), prog,
                    gr.update(visible=show_results), rscore, rfb)

        def get_progress_value_local(s):
            if s.get("stage") == "results": return 100.0
            ri = s.get("current_round_index",0)
            qi = s.get("current_question_index",0)
            return min(((ri*QUESTIONS_PER_ROUND+qi)/total_questions())*100, 99.9)

        submit_btn.click(
            on_submit,
            inputs=[answer_input, session_state],
            outputs=[session_state, chatbot, answer_input, score_display,
                     progress_html, results_panel, results_score_md, results_feedback_md]
        )
        answer_input.submit(
            on_submit,
            inputs=[answer_input, session_state],
            outputs=[session_state, chatbot, answer_input, score_display,
                     progress_html, results_panel, results_score_md, results_feedback_md]
        )

        # Restart
        def on_restart():
            return (make_fresh_session(), [],
                    gr.update(visible=True), gr.update(visible=False),
                    gr.update(visible=False), gr.update(visible=False),
                    gr.update(visible=True), "_Start the interview to see your live score_",
                    "", "")

        restart_btn.click(
            on_restart,
            outputs=[session_state, chatbot,
                     welcome_panel, reg_panel, begin_btn, results_panel,
                     interview_panel, score_display, results_score_md, results_feedback_md]
        )

        # Admin
        refresh_btn.click(get_admin_summary, outputs=[admin_md])
        export_json_btn.click(lambda: json.dumps(load_all_results(), indent=2) or "{}", outputs=[json_box])
        search_btn.click(search_candidate, inputs=[search_input], outputs=[search_results])

    return demo


print("✅ App builder function defined")


# ══════════════════════════════════════════════════════════════════════
# ── CELL 10 ── LAUNCH (Run this last cell to start the app)
# ══════════════════════════════════════════════════════════════════════

print("\n" + "═"*55)
print("  🤖  AI Interview Assistant — Launching...")
print("═"*55)
print(f"  📚 Questions : {len(QUESTION_BANK)}")
print(f"  🧠 spaCy NLP : {'✅ Active' if SPACY_AVAILABLE else '⚠️  Keyword fallback'}")
print(f"  💾 Storage   : {RESULTS_FILE}")
print("═"*55)

app = build_app()

app.launch(
    share=True,            # ← Creates a public gradio.live link for Colab
    debug=False,
    show_error=True,
    quiet=False,
)

✅ spaCy loaded successfully
✅ Loaded 45 questions across 4 rounds
✅ NLP Evaluation Engine ready
✅ Feedback generator & storage ready
✅ Interview flow controller ready
✅ Admin dashboard helpers ready
✅ CSS styles loaded
✅ App builder function defined

═══════════════════════════════════════════════════════
  🤖  AI Interview Assistant — Launching...
═══════════════════════════════════════════════════════
  📚 Questions : 45
  🧠 spaCy NLP : ✅ Active
  💾 Storage   : data/results.json
═══════════════════════════════════════════════════════


/tmp/ipykernel_1862/2950763813.py:733: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="AI Interview Assistant", theme=gr.themes.Base()) as demo:
/tmp/ipykernel_1862/2950763813.py:733: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="AI Interview Assistant", theme=gr.themes.Base()) as demo:
/tmp/ipykernel_1862/2950763813.py:797: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1862/2950763813.py:797: DeprecationWarning: The

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4efcf8cd7daf174872.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
